<a href="https://colab.research.google.com/github/suphakornwong/Chapter_4/blob/main/ImageClassification_Traing_Method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Things to Prepare Before Coding

1. Yolo Pretraind model downloaded from https://docs.ultralytics.com/tasks/classify (or other version)
2. API key form Roboflow platform https://roboflow.com/

Code Cell-A

In [1]:
!pip install ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 8.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ultralytics]


Code Cell-B

In [2]:
import cv2
import torch
import torchvision

print(cv2.__version__, torch.__version__, torchvision.__version__, torch.cuda.is_available())

5.0.0 2.14.1+cpu 0.29.1+cpu False


Code Cell-C

In [3]:
import os
HOME = os.getcwd()
print(HOME)
%mkdir -p {HOME}/datasets
%cd {HOME}/datasets

/content
/content/datasets


Code Cell-D # Upload from your computer

In [ ]:
!unzip /content/PestDataBase.zip

Code Cell-E ## This cell is API key from Roboflow platform

In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="WMKicsfQzlWkd7FdkY5V")
project = rf.workspace("phuwadorn-oeu02").project("snail_pest")
version = project.version(2)
dataset = version.download("folder")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 81.8 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Snail_Pest-2 in folder:: 100%|██████████| 11826/11826 [00:04<00:00, 2370.01it/s]


Code Cell-F # Save your result to google drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/Classification'
os.makedirs(folder_path, exist_ok=True)
print(f"Folder created at: {folder_path}")
save_dir = '/content/drive/MyDrive/Classification'

Mounted at /content/drive
Folder created at: /content/drive/MyDrive/Classification


Code Cell-G

In [6]:
import shutil
import random
from tqdm import tqdm

def split_dataset(dataset_dir, output_dir, train_ratio=0.75, val_ratio=0.15, test_ratio=0.1, seed=42):
   assert train_ratio + val_ratio + test_ratio == 1.0, "Ratios must sum to 1"

   random.seed(seed)
   classes = os.listdir(dataset_dir)

   for class_name in classes:
       class_dir = os.path.join(dataset_dir, class_name)
       if not os.path.isdir(class_dir):
           continue

       images = os.listdir(class_dir)
       random.shuffle(images)

       total = len(images)
       train_count = int(train_ratio * total)
       val_count = int(val_ratio * total)

       train_files = images[:train_count]
       val_files = images[train_count:train_count + val_count]
       test_files = images[train_count + val_count:]

       # Copy files
       for split, split_files in zip(['train', 'val', 'test'], [train_files, val_files, test_files]):
           split_class_dir = os.path.join(output_dir, split, class_name) # Removed the extra 'data' subdirectory
           os.makedirs(split_class_dir, exist_ok=True)

           for file in tqdm(split_files, desc=f"Copying {split}/{class_name}"):
               src = os.path.join(class_dir, file)
               dst = os.path.join(split_class_dir, file)
               if os.path.isfile(src):  # Add this check
                   shutil.copy2(src, dst)

Code Cell-(I-D)

In [9]:
from ultralytics import YOLO

model = YOLO('/content/yolo26s-cls.pt')      ## Right-click at Pretraind model (for example, yolo8n.pt) in /content, Select "copy path"

model.train(
 data='/content/datasets/Snail_Pest-2',    ## find "data.yaml" in datasets folder, Right-click and Select "copy path"
 task='classify',
 epochs=100,
 imgsz=640,
 device=0     ## Set runtime as T4 (GPU) if you want to use "device=0"
)

Ultralytics 8.4.218 🚀 Python-3.13.18 torch-2.14.1+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.
